# In Silico KO: Local WRT Quintile Analysis

Strategy:
1. Predict genome-wide WRT (WT) to establish Q1–Q5 quintile thresholds.
2. Shuffle the KO region, predict only the affected windows (those overlapping the KO region).
3. Classify each affected bin by its WT WRT quintile, then compare ΔWRT.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import torch
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from pyfaidx import Fasta

from src.infer._utils import load_model, parse_bed, rc_average
from src.data.data_utils import one_hot_encode

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
CHECKPOINT  = 'outputs/checkpoints/best_model_0518.pt'
CONFIG      = 'src/configs/transformer_wt.yaml'
FASTA       = '/liaozizhuo/repli-ATAC-seq/data/genomes/rice_all_genomes_v7.fasta'
SPECIES     = 'rice'

# KO target — provide ONE of the two options:
KO_REGION   = 'chr07:4018265-4029484'  # single region, IGV-style (commas OK)
KO_BED      = None                     # BED of summits/peaks; set to None to use KO_REGION
SUMMIT_FLANK = 100                     # only used when KO_BED is provided (summit ± bp)

# Genome-wide WT prediction (for quintile thresholds)
# If you already have a saved CSV from a previous run, set this path to skip re-prediction
WT_CSV      = None   # e.g. 'outputs/ko_wrt/genome_wide_wt.csv'

CHROMS      = [f'chr{i:02d}' for i in range(1, 13)]

RC          = True
BATCH_SIZE  = 16
SEED        = 42

OUTPUT_DIR  = 'outputs/ko_wrt'

# Genome geometry (must match training config)
BIN_SIZE    = 128
CROP_BINS   = 320
CROP_BP     = CROP_BINS * BIN_SIZE   # 40960
OUT_BINS    = 896
STRIDE      = OUT_BINS * BIN_SIZE    # 114688 — gapless

In [ ]:
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, cfg = load_model(CHECKPOINT, CONFIG, device)
window_size = cfg['data']['input_window_length']

fa = Fasta(FASTA)
chrom_sizes = {c: len(fa[c]) for c in fa.keys()}
print(f'Device: {device} | window={window_size} bp | stride={STRIDE} bp')

In [ ]:
# ── Shared helpers ────────────────────────────────────────────────────────────
def parse_region(region_str):
    chrom, rest = region_str.split(':')
    start, end = rest.replace(',', '').split('-')
    return chrom, int(start), int(end)

def fetch_window(seq_bytes, win_start):
    chunk = seq_bytes[win_start: win_start + window_size]
    seq = chunk.decode('ascii')
    if len(seq) < window_size:
        seq = seq + 'N' * (window_size - len(seq))
    return one_hot_encode(seq)  # [4, L]

def predict_batch(batch_oh):
    x = torch.tensor(np.stack(batch_oh), dtype=torch.float32).to(device)
    if RC:
        return rc_average(model, x, SPECIES).cpu().numpy()
    with torch.no_grad():
        return model(x, head=SPECIES)['rt_signals'].cpu().numpy()

def log1p_to_wrt(log1p_pred):
    """[B, T, 4] log1p → [B, T] WRT."""
    pred = np.expm1(np.clip(log1p_pred, 0, None))
    eps = 1e-6
    es, ms, ls = pred[..., 1], pred[..., 2], pred[..., 3]
    return (0.5 * ms + ls) / (es + ms + ls + eps)

In [ ]:
# ── Step 1: Genome-wide WT prediction → quintile thresholds ──────────────────
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

if WT_CSV is not None:
    print(f'Loading WT WRT from {WT_CSV}')
    wt_all = pd.read_csv(WT_CSV)['wrt_wt'].values
else:
    print('Running genome-wide WT prediction...')
    genome_seq = {c: bytearray(str(fa[c]).upper().encode()) for c in CHROMS}
    wt_values = []
    for chrom in CHROMS:
        clen = chrom_sizes.get(chrom)
        if clen is None:
            continue
        windows = range(0, clen - window_size + 1, STRIDE)
        batch_oh, batch_starts = [], []
        def flush_wt():
            if not batch_oh:
                return
            wrt = log1p_to_wrt(predict_batch(batch_oh))  # [B, T]
            wt_values.extend(wrt.ravel().tolist())
            batch_oh.clear(); batch_starts.clear()
        for ws in windows:
            batch_oh.append(fetch_window(genome_seq[chrom], ws))
            batch_starts.append(ws)
            if len(batch_oh) == BATCH_SIZE:
                flush_wt()
        flush_wt()
        print(f'  {chrom}: done')
    wt_all = np.array(wt_values)
    pd.DataFrame({'wrt_wt': wt_all}).to_csv(out_dir / 'genome_wide_wt.csv', index=False)
    print(f'Saved WT WRT to {out_dir / "genome_wide_wt.csv"}  (set WT_CSV to reuse)')

# compute quintile bin edges from genome-wide WT distribution
quintile_edges = np.percentile(wt_all, [0, 20, 40, 60, 80, 100])
print(f'Quintile edges: {quintile_edges.round(4)}')

In [ ]:
# ── Step 2: Build KO genome, predict only affected windows ───────────────────
print('Loading genome into memory...')
genome_wt = {c: bytearray(str(fa[c]).upper().encode()) for c in CHROMS}
genome_ko = {c: bytearray(genome_wt[c]) for c in CHROMS}

def shuffle_interval(chrom, start, end):
    region = bytes(genome_ko[chrom][start:end])
    shuffled = bytearray(np.random.permutation(list(region)).tolist())
    genome_ko[chrom][start:end] = shuffled

# Collect all KO intervals and apply shuffles
ko_intervals = []  # list of (chrom, start, end)
if KO_BED is not None:
    for chrom, start, end, name, strand in parse_bed(KO_BED):
        if chrom not in genome_ko:
            continue
        summit = (start + end) // 2
        s = max(0, summit - SUMMIT_FLANK)
        e = min(len(genome_ko[chrom]), summit + SUMMIT_FLANK)
        shuffle_interval(chrom, s, e)
        ko_intervals.append((chrom, s, e))
    label = f'{Path(KO_BED).name} ({len(ko_intervals)} regions)'
else:
    ko_chrom, ko_start, ko_end = parse_region(KO_REGION)
    shuffle_interval(ko_chrom, ko_start, ko_end)
    ko_intervals.append((ko_chrom, ko_start, ko_end))
    label = KO_REGION

print(f'KO: {label}')

# Find all windows that overlap any KO interval
affected = set()
for ko_chrom, ko_start, ko_end in ko_intervals:
    clen = chrom_sizes.get(ko_chrom, 0)
    all_starts = range(0, clen - window_size + 1, STRIDE)
    for ws in all_starts:
        # window output region: [ws + CROP_BP, ws + CROP_BP + OUT_BINS * BIN_SIZE)
        # but the full sequence context is [ws, ws + window_size)
        if ws < ko_end and ws + window_size > ko_start:
            affected.add((ko_chrom, ws))

print(f'Affected windows: {len(affected)}')

In [ ]:
# ── Predict WT and KO for affected windows only ───────────────────────────────
records = []
affected_list = sorted(affected)

batch_wt, batch_ko, batch_meta = [], [], []

def flush_ko():
    if not batch_wt:
        return
    wrt_wt = log1p_to_wrt(predict_batch(batch_wt))  # [B, T]
    wrt_ko = log1p_to_wrt(predict_batch(batch_ko))
    for i, (chrom, ws) in enumerate(batch_meta):
        for k in range(OUT_BINS):
            bs = ws + CROP_BP + k * BIN_SIZE
            records.append({
                'chrom': chrom,
                'bin_start': bs,
                'wrt_wt': float(wrt_wt[i, k]),
                'wrt_ko': float(wrt_ko[i, k]),
            })
    batch_wt.clear(); batch_ko.clear(); batch_meta.clear()

for chrom, ws in affected_list:
    batch_wt.append(fetch_window(genome_wt[chrom], ws))
    batch_ko.append(fetch_window(genome_ko[chrom], ws))
    batch_meta.append((chrom, ws))
    if len(batch_wt) == BATCH_SIZE:
        flush_ko()
flush_ko()

df = pd.DataFrame(records)
print(f'Affected bins: {len(df)}')

In [ ]:
# ── Step 3: Classify bins by genome-wide quintile thresholds ─────────────────
df['delta_wrt'] = df['wrt_ko'] - df['wrt_wt']

# assign quintile using genome-wide edges (not local qcut)
df['quintile'] = pd.cut(
    df['wrt_wt'],
    bins=quintile_edges,
    labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'],
    include_lowest=True
)

summary = df.groupby('quintile', observed=True)['delta_wrt'].agg(
    n='count', mean='mean', sem=lambda x: x.sem()
).reset_index()
print(summary.to_string(index=False))

In [ ]:
# ── Bar plot ──────────────────────────────────────────────────────────────────
colors = ['#2166ac', '#74add1', '#ffffbf', '#f46d43', '#d73027']

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(
    summary['quintile'], summary['mean'],
    yerr=summary['sem'], capsize=4,
    color=colors, edgecolor='black', linewidth=0.8
)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('WRT quintile (genome-wide WT)', fontsize=11)
ax.set_ylabel('ΔWRT  (KO − WT)', fontsize=11)
ax.set_title(f'In silico KO: {label}\n{len(df):,} affected bins', fontsize=10)
plt.tight_layout()
fig.savefig(out_dir / 'ko_wrt_quintile.pdf', bbox_inches='tight')
plt.show()
print(f'Saved: {out_dir / "ko_wrt_quintile.pdf"}')